# 02 Multi Sample Inference v0.1

**Notebook version:** v0.1  
**Category:** inference  
**Purpose:** Select one ROI-FCN run, one distance-orientation regression run, and one raw synthetic corpus, then execute a multi-sample inference pass over a corpus slice using the new model-driven ROI path.  
**Inputs:** `./models`, `./input`, `./src/inference_v0_1`  
**Expected outputs:** tabular inference summaries for the requested corpus slice, plus optional JSON save artifacts.  
**Artifact write mode:** optional aggregated JSON under `./output/<model-name>/inference-output_<model-name>.json`


In [1]:
# Repo Setup
from pathlib import Path
import sys


INFERENCE_ROOT_CANDIDATES = ('05_inference-v0.2', '05_inference-v0.1', '04_inference-v0.1')


def find_repo_root(start: Path | None = None) -> tuple[Path, Path]:
    candidate = (start or Path.cwd()).resolve()
    if candidate.is_file():
        candidate = candidate.parent

    for current in (candidate, *candidate.parents):
        for name in INFERENCE_ROOT_CANDIDATES:
            inference_root = current / name
            if (inference_root / 'src').exists() and (inference_root / 'models').exists():
                return current, inference_root

    raise RuntimeError(
        f'Could not locate repo root containing any of: {INFERENCE_ROOT_CANDIDATES}'
    )


repo_root, inference_root = find_repo_root()
src_root = inference_root / 'src'
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

print(f'repo_root={repo_root}')
print(f'inference_root={inference_root}')


repo_root=/home/mitch/development/raccoon-ball
inference_root=/home/mitch/development/raccoon-ball/05_inference-v0.2


In [2]:
# Control Pane
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display

from inference_v0_1 import (
    default_raw_corpus_roots,
    discover_model_runs,
    discover_raw_corpora,
    load_corpus_samples,
    run_multi_sample_inference,
)

all_models = discover_model_runs(inference_root / 'models')
if not all_models:
    raise FileNotFoundError(f'No model run artifacts found under {inference_root / "models"}')

distance_models = [artifact for artifact in all_models if artifact.model_family == 'distance-orientation']
roi_models = [artifact for artifact in all_models if artifact.model_family == 'roi-fcn']
if not distance_models:
    raise FileNotFoundError('No distance-orientation model runs found under ./models/distance-orientation')
if not roi_models:
    raise FileNotFoundError('No ROI-FCN model runs found under ./models/roi-fcn')

raw_corpus_roots = default_raw_corpus_roots()
corpora = discover_raw_corpora()
if not corpora:
    searched_roots = '\n  - '.join(str(path) for path in raw_corpus_roots)
    raise FileNotFoundError(
        'No valid raw-image corpora found in any configured raw-corpus root:\n'
        f'  - {searched_roots}\n'
        'Expected corpus folders with images/ plus manifests/run.json and manifests/samples.csv.'
    )

corpus_names = [corpus.name for corpus in corpora]
duplicate_corpus_names = sorted({name for name in corpus_names if corpus_names.count(name) > 1})
if duplicate_corpus_names:
    raise ValueError(
        f'Duplicate corpus names found across raw-corpus roots: {duplicate_corpus_names}'
    )

distance_model_by_label = {artifact.label: artifact for artifact in distance_models}
roi_model_by_label = {artifact.label: artifact for artifact in roi_models}
corpus_by_name = {corpus.name: corpus for corpus in corpora}
corpus_samples_cache: dict[str, pd.DataFrame] = {}
local_input_root = inference_root / 'input'
all_input_dirs = sorted(path.name for path in local_input_root.iterdir() if path.is_dir()) if local_input_root.exists() else []
ignored_dirs = sorted(set(all_input_dirs).difference(corpus_by_name))


def selected_model_output_dir_name(model_artifact) -> str:
    run_dir = Path(model_artifact.run_dir).resolve()
    if run_dir.name.startswith('run_') and run_dir.parent.name == 'runs' and run_dir.parent.parent.name:
        return run_dir.parent.parent.name
    return run_dir.name


print(f'discovered_models_total={len(all_models)}')
print(f'discovered_distance_orientation_models={len(distance_models)}')
print(f'discovered_roi_fcn_models={len(roi_models)}')
print(f'discovered_raw_corpora={len(corpora)}')
print('raw_corpus_roots=', [str(path) for path in raw_corpus_roots if path.exists()])
if ignored_dirs:
    print('ignored_non_raw_input_dirs=', ignored_dirs)


def get_corpus_samples(corpus_name: str) -> pd.DataFrame:
    if corpus_name not in corpus_samples_cache:
        corpus_samples_cache[corpus_name] = load_corpus_samples(corpus_by_name[corpus_name])
    return corpus_samples_cache[corpus_name]


distance_model_select = widgets.Select(
    options=[artifact.label for artifact in distance_models],
    value=distance_models[0].label,
    description='Regressor',
    rows=min(8, max(4, len(distance_models))),
)
roi_model_select = widgets.Select(
    options=[artifact.label for artifact in roi_models],
    value=roi_models[-1].label,
    description='ROI-FCN',
    rows=min(8, max(4, len(roi_models))),
)
corpus_select = widgets.Select(
    options=[corpus.name for corpus in corpora],
    value=corpora[0].name,
    description='Corpus',
    rows=min(8, max(4, len(corpora))),
)
image_select = widgets.SelectMultiple(
    options=[],
    value=(),
    description='Image',
    rows=12,
    layout=widgets.Layout(height='280px'),
)
sample_count_input = widgets.IntText(
    value=1,
    description='# of Samples',
)
offset_input = widgets.IntText(
    value=0,
    description='Offset',
)
run_button = widgets.Button(
    description='Run Inference',
    button_style='primary',
    disabled=True,
)
save_toggle = widgets.Checkbox(value=False, description='Save JSON')
corpus_count_html = widgets.HTML()
selection_summary_html = widgets.HTML()
status_html = widgets.HTML()
results_out = widgets.Output()


def clear_results() -> None:
    status_html.value = ''
    with results_out:
        clear_output()


def update_run_state() -> None:
    corpus_name = corpus_select.value
    if not corpus_name:
        corpus_count_html.value = ''
        selection_summary_html.value = ''
        run_button.disabled = True
        return

    samples_df = get_corpus_samples(corpus_name)
    total_samples = int(len(samples_df))
    corpus_count_html.value = f'<b>Total samples in corpus:</b> {total_samples}'

    offset_value = int(offset_input.value)
    sample_count_value = int(sample_count_input.value)

    if total_samples <= 0:
        selection_summary_html.value = '<span style="color:#b00020;"><b>No selectable samples are available in this corpus.</b></span>'
        run_button.disabled = True
        return
    if offset_value < 0:
        selection_summary_html.value = '<span style="color:#b00020;"><b>Offset must be 0 or greater.</b></span>'
        run_button.disabled = True
        return
    if sample_count_value <= 0:
        selection_summary_html.value = '<span style="color:#b00020;"><b># of Samples must be greater than 0.</b></span>'
        run_button.disabled = True
        return
    if offset_value >= total_samples:
        selection_summary_html.value = (
            f'<span style="color:#b00020;"><b>Offset is out of range.</b> '
            f'Choose a value between 0 and {total_samples - 1}.</span>'
        )
        run_button.disabled = True
        return

    actual_count = min(sample_count_value, total_samples - offset_value)
    range_end = offset_value + actual_count - 1
    truncation_note = ''
    if actual_count < sample_count_value:
        truncation_note = (
            f' Requested {sample_count_value}; only {actual_count} sample(s) are available '
            f'from offset {offset_value}.'
        )

    selection_summary_html.value = (
        f'<b>Run slice:</b> {actual_count} sample(s), indices {offset_value} to {range_end}. '
        '<i>Image selection is displayed for reference only and is ignored when running inference.</i>'
        f'{truncation_note}'
    )
    run_button.disabled = False


def refresh_corpus_options(*_args) -> None:
    clear_results()
    corpus_name = corpus_select.value
    if not corpus_name:
        image_select.options = []
        image_select.value = ()
        update_run_state()
        return

    samples_df = get_corpus_samples(corpus_name)
    image_select.options = samples_df['__image_name__'].tolist()
    image_select.value = ()
    update_run_state()


def on_model_change(change) -> None:
    if change['name'] == 'value':
        clear_results()


def on_corpus_change(change) -> None:
    if change['name'] == 'value':
        refresh_corpus_options()


def on_numeric_change(change) -> None:
    if change['name'] != 'value':
        return
    if change['new'] != change['old']:
        clear_results()
    update_run_state()


def on_image_change(change) -> None:
    if change['name'] != 'value':
        return
    if change['new'] != change['old']:
        clear_results()
    update_run_state()


def on_run_clicked(_button) -> None:
    clear_results()
    selected_distance_model = distance_model_by_label[distance_model_select.value]
    selected_roi_model = roi_model_by_label[roi_model_select.value]
    selected_corpus = corpus_by_name[corpus_select.value]
    samples_df = get_corpus_samples(selected_corpus.name)
    total_samples = int(len(samples_df))
    offset_value = int(offset_input.value)
    sample_count_value = int(sample_count_input.value)

    if total_samples <= 0:
        status_html.value = '<span style="color:#b00020;"><b>No selectable samples are available in the selected corpus.</b></span>'
        return
    if offset_value < 0:
        status_html.value = '<span style="color:#b00020;"><b>Offset must be 0 or greater.</b></span>'
        return
    if sample_count_value <= 0:
        status_html.value = '<span style="color:#b00020;"><b># of Samples must be greater than 0.</b></span>'
        return
    if offset_value >= total_samples:
        status_html.value = (
            f'<span style="color:#b00020;"><b>Offset is out of range.</b> '
            f'Choose a value between 0 and {total_samples - 1}.</span>'
        )
        return

    requested_count = sample_count_value
    actual_count = min(requested_count, total_samples - offset_value)
    save_result = bool(save_toggle.value)
    results_root_path = None
    if save_result:
        results_root_path = inference_root / 'output' / selected_model_output_dir_name(selected_distance_model)

    run_button.disabled = True
    status_html.value = f'<i>Running inference for {actual_count} sample(s)...</i>'
    try:
        results = run_multi_sample_inference(
            selected_distance_model.run_dir,
            selected_corpus.root,
            roi_model_run_dir=selected_roi_model.run_dir,
            offset=offset_value,
            num_samples=requested_count,
            save_result=save_result,
            results_root_path=results_root_path,
            device='cuda',
        )
    except Exception as exc:
        status_html.value = (
            f'<span style="color:#b00020;"><b>Inference failed:</b> {exc}</span>'
        )
    else:
        processed_count = len(results)
        truncation_note = ''
        if processed_count < requested_count:
            truncation_note = (
                f' Requested {requested_count}; processed {processed_count} available sample(s).'
            )
        status_html.value = (
            f'<b>Inference completed.</b> Processed {processed_count} sample(s) starting at offset {offset_value}.'
            f'{truncation_note}'
        )

        detail_rows = [
            {
                'sample_index': offset_value + index,
                'image': result.selected_image_name,
                'sample_id': result.sample_id,
                'predicted_distance_m': result.predicted_distance_m,
                'actual_distance_m': result.actual_distance_m,
                'absolute_distance_error_m': result.absolute_distance_error_m,
                'predicted_orientation_deg': result.predicted_orientation_deg,
                'actual_orientation_deg': result.actual_orientation_deg,
                'absolute_orientation_error_deg': result.absolute_orientation_error_deg,
            }
            for index, result in enumerate(results)
        ]
        detail_df = pd.DataFrame(detail_rows)
        summary_rows = [
            {'field': 'Distance model', 'value': results[0].selected_model_label},
            {'field': 'ROI-FCN model', 'value': results[0].selected_roi_model_label},
            {'field': 'Corpus', 'value': results[0].selected_corpus_name},
            {'field': 'Offset', 'value': offset_value},
            {'field': 'Requested samples', 'value': requested_count},
            {'field': 'Processed samples', 'value': processed_count},
            {
                'field': 'Mean absolute distance error (m)',
                'value': f'{detail_df["absolute_distance_error_m"].mean():.4f}',
            },
            {
                'field': 'Mean absolute orientation error (deg)',
                'value': f'{detail_df["absolute_orientation_error_deg"].mean():.4f}',
            },
            {
                'field': 'Max absolute distance error (m)',
                'value': f'{detail_df["absolute_distance_error_m"].max():.4f}',
            },
            {
                'field': 'Max absolute orientation error (deg)',
                'value': f'{detail_df["absolute_orientation_error_deg"].max():.4f}',
            },
        ]

        with results_out:
            clear_output()
            display(pd.DataFrame(summary_rows))
            display(
                detail_df.round(
                    {
                        'predicted_distance_m': 4,
                        'actual_distance_m': 4,
                        'absolute_distance_error_m': 4,
                        'predicted_orientation_deg': 4,
                        'actual_orientation_deg': 4,
                        'absolute_orientation_error_deg': 4,
                    }
                )
            )
            if save_result and results and results[0].saved_json_path is not None:
                print(f'Saved JSON: {results[0].saved_json_path}')
                print(f'Saved ROI images: {processed_count}')
    finally:
        update_run_state()


distance_model_select.observe(on_model_change, names='value')
roi_model_select.observe(on_model_change, names='value')
corpus_select.observe(on_corpus_change, names='value')
sample_count_input.observe(on_numeric_change, names='value')
offset_input.observe(on_numeric_change, names='value')
image_select.observe(on_image_change, names='value')
run_button.on_click(on_run_clicked)

refresh_corpus_options()

controls = widgets.VBox(
    [
        widgets.HBox([distance_model_select, roi_model_select, corpus_select, image_select]),
        widgets.HBox([sample_count_input, offset_input, run_button, save_toggle]),
        corpus_count_html,
        selection_summary_html,
        status_html,
        results_out,
    ]
)
display(controls)


discovered_models_total=4
discovered_distance_orientation_models=1
discovered_roi_fcn_models=3
discovered_raw_corpora=1
raw_corpus_roots= ['/home/mitch/development/raccoon-ball/05_inference-v0.2/input', '/home/mitch/development/raccoon-ball/02_synthetic-data-processing-v3.0/input-images']
ignored_non_raw_input_dirs= ['26-04-11_v021-validate-shuffled-images']
